# AWS Glue Daily Incremental Bronze to Silver

This notebook processes the new daily files from S3 `bronze/`, merges them with existing S3 `silver/` data, removes duplicates, then overwrites the silver outputs as both Parquet and CSV.

Daily input pattern:

```text
s3://cyber-threat-intel-data-lake/bronze/assets/YYYY-MM-DD/asset_inventory.csv
s3://cyber-threat-intel-data-lake/bronze/cisa_kev/YYYY-MM-DD/cisa_kev.json
s3://cyber-threat-intel-data-lake/bronze/epss/YYYY-MM-DD/epss_scores.json
s3://cyber-threat-intel-data-lake/bronze/nvd/YYYY-MM-DD/nvd_cves.json
```

Silver output pattern:

```text
s3://cyber-threat-intel-data-lake/silver/asset_inventory_clean/
s3://cyber-threat-intel-data-lake/silver/cisa_kev_clean/
s3://cyber-threat-intel-data-lake/silver/epss_scores/
s3://cyber-threat-intel-data-lake/silver/cves_clean/
```


## 1. Glue Session Setup

Run this first. If your role ARN is different, change only the `%iam_role` line.

In [7]:
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5
%idle_timeout 30
%iam_role arn:aws:iam::311414083183:role/AWSGlueServiceRole-CyberThreatIntel
%additional_python_modules pyarrow

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Current iam_role is arn:aws:iam::311414083183:role/AWSGlueServiceRole-CyberThreatIntel
iam_role has been set to arn:aws:iam::311414083183:role/AWSGlueServiceRole-CyberThreatIntel.
Additional python modules to be included:
pyarrow


## 2. Imports and Paths

Change `run_date` every day to match the new bronze folder.

In [1]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")

bucket = "cyber-threat-intel-data-lake"
bronze_base = f"s3://{bucket}/bronze"
silver_base = f"s3://{bucket}/silver"

# Change this date every day.
from datetime import datetime , timedelta
from zoneinfo import ZoneInfo

run_date = (datetime.now(ZoneInfo("Africa/Cairo")) - timedelta(days=1)).strftime("%Y-%m-%d")

paths = {
    "assets_bronze": f"{bronze_base}/assets/{run_date}/asset_inventory.csv",
    "cisa_bronze": f"{bronze_base}/cisa_kev/{run_date}/cisa_kev.json",
    "epss_bronze": f"{bronze_base}/epss/{run_date}/epss_scores.json",
    "epss_initial_bronze": f"{bronze_base}/epss/initial/epss_scores_full.json",
    "cves_bronze": f"{bronze_base}/nvd/initial/nvd_cves_full.json",

    "assets_silver": f"{silver_base}/asset_inventory_clean/",
    "cisa_silver": f"{silver_base}/cisa_kev_clean/",
    "epss_silver": f"{silver_base}/epss_scores/",
    "cves_silver": f"{silver_base}/cves_clean/",

    "assets_csv": f"{silver_base}/asset_inventory_clean_csv/",
    "cisa_csv": f"{silver_base}/cisa_kev_clean_csv/",
    "epss_csv": f"{silver_base}/epss_scores_csv/",
    "cves_csv": f"{silver_base}/cves_clean_csv/",
    "cve_affected_products_silver": f"{silver_base}/cve_affected_products/",
    "cve_affected_products_csv": f"{silver_base}/cve_affected_products_csv/",
}
paths

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 30
Session ID: acdcded5-5668-47aa-9752-3458d63fd7b0
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--additional-python-modules pyarrow
Waiting for session acdcded5-5668-47aa-9752-3458d63fd7b0 to get into ready status...
Session acdcded5-5668-47aa-9752-3458d63fd7b0 has been created.
{'assets_bronze': 's3://cyber-threat-intel-data-lake/bronze/assets/2026-06-07/asset_inventory.csv', 'cisa_bronze': 's3://cyber-threat-intel-data-lake/bronze/cisa_kev/2026-06-07/cisa_kev.json', 'epss_bronze': 's3://cyber-threat-intel-data-lake/bronze/epss/2026-06-07/epss_scores.json', 'epss_initial_bronze': 's3://cyber-threat-intel-data-lake/bronze/epss/initial/epss_scores_full.json', 'cves_bronze': 's3://cyber-threat-intel-data-lake/bronze/nvd/initial/nvd_cves_full.json', 'assets_silver': 's3://cyber-threat-intel-data-lake/silver/ass

## 3. Helper Functions

In [2]:
def read_existing_silver(path, schema):
    """Read existing silver Parquet, or return an empty DataFrame if it does not exist yet."""
    try:
        df = spark.read.parquet(path)
        print(f"Existing silver rows at {path}: {df.count()}")
        return df
    except Exception:
        print(f"No existing silver data found at {path}. Starting from empty DataFrame.")
        return spark.createDataFrame([], schema)


def merge_overwrite_silver(new_df, silver_path, csv_path, duplicate_keys, dataset_name):
    """
    Merge new daily data with old silver, deduplicate, then overwrite Parquet and CSV safely.

    Important: do not read from and overwrite the same S3 path in one lazy Spark lineage.
    We first write the merged result to a staging folder, then read staging and overwrite final silver.
    """
    old_df = read_existing_silver(silver_path, new_df.schema)

    updated_df = (
        old_df
        .unionByName(new_df, allowMissingColumns=True)
        .dropDuplicates(duplicate_keys)
    )

    staging_path = f"{silver_base}/_staging/{dataset_name}/{run_date}/"

    updated_df.write.mode("overwrite").parquet(staging_path)
    staged_df = spark.read.parquet(staging_path)

    updated_count = staged_df.count()
    print(f"Updated rows for {dataset_name}: {updated_count}")

    staged_df.write.mode("overwrite").parquet(silver_path)
    staged_df.write.mode("overwrite").option("header", "true").csv(csv_path)

    print("Saved staging:", staging_path)
    print("Saved Parquet:", silver_path)
    print("Saved CSV:", csv_path)
    return staged_df

## 4. Asset Inventory Daily Merge

In [3]:
assets_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(paths["assets_bronze"])
)

assets_clean = assets_raw.select([
    F.col(c).alias(c.strip().lower().replace(" ", "_"))
    for c in assets_raw.columns
])

if "vendor" in assets_clean.columns:
    assets_clean = assets_clean.withColumn("vendor", F.lower(F.trim(F.col("vendor"))))

if "product" in assets_clean.columns:
    assets_clean = assets_clean.withColumn("product", F.lower(F.trim(F.col("product"))))

# Use asset_id if it exists; otherwise deduplicate by all columns.
asset_duplicate_keys = ["asset_id"] if "asset_id" in assets_clean.columns else assets_clean.columns

assets_updated = merge_overwrite_silver(
    new_df=assets_clean,
    silver_path=paths["assets_silver"],
    csv_path=paths["assets_csv"],
    duplicate_keys=asset_duplicate_keys,
    dataset_name="assets",
)
assets_updated.show(10, truncate=False)

Existing silver rows at s3://cyber-threat-intel-data-lake/silver/asset_inventory_clean/: 5000
Updated rows for assets: 5000
Saved staging: s3://cyber-threat-intel-data-lake/silver/_staging/assets/2026-06-07/
Saved Parquet: s3://cyber-threat-intel-data-lake/silver/asset_inventory_clean/
Saved CSV: s3://cyber-threat-intel-data-lake/silver/asset_inventory_clean_csv/
+--------+---------------------------+---------+---------------+-------+-----------+---------------+
|asset_id|hostname                   |vendor   |product        |version|criticality|internet_facing|
+--------+---------------------------+---------+---------------+-------+-----------+---------------+
|A00001  |vmware-esxi-1              |vmware   |esxi           |9.2.1  |Low        |No             |
|A00002  |microsoft-exchange-server-2|microsoft|exchange server|1.3.29 |High       |No             |
|A00003  |cisco-meraki-3             |cisco    |meraki         |7.8.82 |Critical   |No             |
|A00004  |linux-ubuntu-4    

cisa_updated = merge_overwrite_silver(
    new_df=cisa_clean,
    silver_path=paths["cisa_silver"],
    csv_path=paths["cisa_csv"],
    duplicate_keys=["cve_id"],
    dataset_name="cisa",
)## 5. CISA KEV Daily Merge

In [5]:
cisa_raw = spark.read.option("multiLine", "true").json(paths["cisa_bronze"])

cisa_clean = (
    cisa_raw
    .select(F.explode("vulnerabilities").alias("v"))
    .select(
        F.col("v.cveID").alias("cve_id"),
        F.lower(F.trim(F.col("v.vendorProject"))).alias("vendor"),
        F.lower(F.trim(F.col("v.product"))).alias("product"),
        F.trim(F.col("v.vulnerabilityName")).alias("vulnerability_name"),
        F.regexp_replace(F.trim(F.col("v.shortDescription")), r"\s+", " ").alias("short_description"),
        F.to_date("v.dateAdded").alias("date_added"),
        F.to_date("v.dueDate").alias("due_date"),
        F.col("v.knownRansomwareCampaignUse").alias("known_ransomware_campaign_use"),
        F.concat_ws(",", F.col("v.cwes")).alias("cwes"),
    )
    .dropDuplicates(["cve_id"])
)

cisa_updated = merge_overwrite_silver(
    new_df=cisa_clean,
    silver_path=paths["cisa_silver"],
    csv_path=paths["cisa_csv"],
    duplicate_keys=["cve_id"],
    dataset_name="cisa",
)

cisa_updated.show(10, truncate=False)


Existing silver rows at s3://cyber-threat-intel-data-lake/silver/cisa_kev_clean/: 1612
Updated rows for cisa: 1612
Saved staging: s3://cyber-threat-intel-data-lake/silver/_staging/cisa/2026-06-07/
Saved Parquet: s3://cyber-threat-intel-data-lake/silver/cisa_kev_clean/
Saved CSV: s3://cyber-threat-intel-data-lake/silver/cisa_kev_clean_csv/
+-------------+--------------------+-----------------------------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+-----------------------------+-------+
|cve_id       |vendor              |product

## 6. EPSS Scores Daily Merge

In [6]:
epss_raw = spark.read.option("multiLine", "true").json(paths["epss_bronze"])

epss_clean = (
    epss_raw
    .select(F.explode("data").alias("d"))
    .select(
        F.col("d.cve").alias("cve_id"),
        F.col("d.epss").cast("double").alias("epss_score"),
        F.col("d.percentile").cast("double").alias("percentile"),
        F.to_date("d.date").alias("score_date"),
    )
    .dropDuplicates(["cve_id", "score_date"])
)

epss_updated = merge_overwrite_silver(
    new_df=epss_clean,
    silver_path=paths["epss_silver"],
    csv_path=paths["epss_csv"],
    duplicate_keys=["cve_id", "score_date"],
    dataset_name="epss",
)

epss_updated.show(10, truncate=False)

Existing silver rows at s3://cyber-threat-intel-data-lake/silver/epss_scores/: 676817
Updated rows for epss: 676817
Saved staging: s3://cyber-threat-intel-data-lake/silver/_staging/epss/2026-06-07/
Saved Parquet: s3://cyber-threat-intel-data-lake/silver/epss_scores/
Saved CSV: s3://cyber-threat-intel-data-lake/silver/epss_scores_csv/
+-------------+----------+----------+----------+
|cve_id       |epss_score|percentile|score_date|
+-------------+----------+----------+----------+
|CVE-2026-9930|4.1E-4    |0.12942   |2026-06-05|
|CVE-2026-9297|0.01409   |0.80863   |2026-06-05|
|CVE-2026-9245|6.4E-4    |0.20226   |2026-06-05|
|CVE-2026-7699|3.4E-4    |0.10403   |2026-06-05|
|CVE-2026-7303|7.4E-4    |0.22579   |2026-06-05|
|CVE-2026-7179|2.0E-4    |0.05887   |2026-06-05|
|CVE-2026-6903|3.5E-4    |0.10887   |2026-06-05|
|CVE-2026-6587|1.4E-4    |0.02936   |2026-06-05|
|CVE-2026-6308|3.9E-4    |0.12232   |2026-06-05|
|CVE-2026-5858|8.8E-4    |0.25238   |2026-06-05|
+-------------+----------+-

## 7. NVD CVEs Daily Merge

This reads the daily NVD file, not the huge initial full file. The first full load should be done once separately. After that, use this daily merge.

In [11]:
from pyspark.sql import functions as F

# -------------------------
# NVD CVEs Daily Merge (Single table with CPE fields)
# -------------------------

raw_cves = spark.read.option("multiLine", "true").json(paths["cves_bronze"])

# Extract first CPE match per CVE
cve_columns = raw_cves.select("cve.*").columns
has_configurations = "configurations" in cve_columns

if has_configurations:
    first_cpe = (
        raw_cves
        .select(
            F.col("cve.id").alias("cve_id"),
            F.col("cve.configurations").alias("configurations")
        )
        .filter(F.col("configurations").isNotNull())
        .select(
            "cve_id",
            F.expr("configurations[0].nodes[0].cpeMatch[0]").alias("first_cpe")
        )
        .filter(F.col("first_cpe").isNotNull())
        .filter(F.col("first_cpe.vulnerable") == True)
        .select(
            "cve_id",
            F.col("first_cpe.criteria").alias("criteria"),
            F.col("first_cpe.matchCriteriaId").alias("matchCriteriaId"),
            # Parse CPE: cpe:2.3:part:vendor:product:version:...
            F.lower(F.regexp_replace(F.split(F.col("first_cpe.criteria"), ":").getItem(3), "_", " ")).alias("vendor"),
            F.lower(F.regexp_replace(F.split(F.col("first_cpe.criteria"), ":").getItem(4), "_", " ")).alias("product"),
            F.split(F.col("first_cpe.criteria"), ":").getItem(5).alias("version"),
        )
    )
else:
    first_cpe = spark.createDataFrame(
        [],
        "cve_id string, criteria string, matchCriteriaId string, vendor string, product string, version string"
    )

# Build cves_clean with CVSS fields
cves_with_metrics = (
    raw_cves
    .withColumn("description", F.expr("filter(cve.descriptions, x -> x.lang = 'en')[0].value"))
    .withColumn("cvss_v40", F.expr("filter(cve.metrics.cvssMetricV40, x -> lower(x.type) = 'primary')[0]"))
    .withColumn("cvss_any_v40", F.expr("cve.metrics.cvssMetricV40[0]"))
    .withColumn("cvss_v31", F.expr("filter(cve.metrics.cvssMetricV31, x -> lower(x.type) = 'primary')[0]"))
    .withColumn("cvss_any_v31", F.expr("cve.metrics.cvssMetricV31[0]"))
    .withColumn("cvss_v2", F.expr("filter(cve.metrics.cvssMetricV2, x -> lower(x.type) = 'primary')[0]"))
    .withColumn("cvss_any_v2", F.expr("cve.metrics.cvssMetricV2[0]"))
)

cves_clean = (
    cves_with_metrics
    .select(
        F.col("cve.id").alias("cve_id"),
        F.regexp_replace(F.trim(F.col("description")), r"\s+", " ").alias("description"),
        F.coalesce(
            F.col("cvss_v40.cvssData.version"),
            F.col("cvss_any_v40.cvssData.version"),
            F.col("cvss_v31.cvssData.version"),
            F.col("cvss_any_v31.cvssData.version"),
            F.col("cvss_v2.cvssData.version"),
            F.col("cvss_any_v2.cvssData.version"),
        ).alias("cvss_version"),
        F.coalesce(
            F.col("cvss_v40.cvssData.vectorString"),
            F.col("cvss_any_v40.cvssData.vectorString"),
            F.col("cvss_v31.cvssData.vectorString"),
            F.col("cvss_any_v31.cvssData.vectorString"),
            F.col("cvss_v2.cvssData.vectorString"),
            F.col("cvss_any_v2.cvssData.vectorString"),
        ).alias("vector_string"),
        F.coalesce(
            F.col("cvss_v40.cvssData.baseScore"),
            F.col("cvss_any_v40.cvssData.baseScore"),
            F.col("cvss_v31.cvssData.baseScore"),
            F.col("cvss_any_v31.cvssData.baseScore"),
            F.col("cvss_v2.cvssData.baseScore"),
            F.col("cvss_any_v2.cvssData.baseScore"),
        ).cast("double").alias("baseScore"),
        F.coalesce(
            F.col("cvss_v40.cvssData.baseSeverity"),
            F.col("cvss_any_v40.cvssData.baseSeverity"),
            F.col("cvss_v31.cvssData.baseSeverity"),
            F.col("cvss_any_v31.cvssData.baseSeverity"),
            F.col("cvss_v2.baseSeverity"),
            F.col("cvss_any_v2.baseSeverity"),
        ).alias("baseSeverity"),
        F.coalesce(
            F.col("cvss_v31.exploitabilityScore"),
            F.col("cvss_any_v31.exploitabilityScore"),
            F.col("cvss_v2.exploitabilityScore"),
            F.col("cvss_any_v2.exploitabilityScore"),
        ).cast("double").alias("exploitabilityScore"),
        F.coalesce(
            F.col("cvss_v31.impactScore"),
            F.col("cvss_any_v31.impactScore"),
            F.col("cvss_v2.impactScore"),
            F.col("cvss_any_v2.impactScore"),
        ).cast("double").alias("impactScore"),
        F.to_timestamp("cve.published").alias("published_date"),
        F.to_timestamp("cve.lastModified").alias("last_modified_date"),
        F.col("cve.vulnStatus").alias("vuln_status"),
    )
    .dropDuplicates(["cve_id"])
)

# Join CPE fields (left join — keeps all CVEs even without CPE)
cves_final = cves_clean.join(first_cpe, on="cve_id", how="left")

print("Final rows:", cves_final.count())
cves_final.printSchema()

# Merge to silver
cves_updated = merge_overwrite_silver(
    new_df=cves_final,
    silver_path=paths["cves_silver"],
    csv_path=paths["cves_csv"],
    duplicate_keys=["cve_id"],
    dataset_name="cves",
)

cves_updated.select("cve_id", "baseScore", "vendor", "product", "version", "criteria").show(10, truncate=False)
print("nvd done")

Final rows: 355074
root
 |-- cve_id: string (nullable = true)
 |-- description: string (nullable = true)
 |-- cvss_version: string (nullable = true)
 |-- vector_string: string (nullable = true)
 |-- baseScore: double (nullable = true)
 |-- baseSeverity: string (nullable = true)
 |-- exploitabilityScore: double (nullable = true)
 |-- impactScore: double (nullable = true)
 |-- published_date: timestamp (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- vuln_status: string (nullable = true)
 |-- criteria: string (nullable = true)
 |-- matchCriteriaId: string (nullable = true)
 |-- vendor: string (nullable = true)
 |-- product: string (nullable = true)
 |-- version: string (nullable = true)

Existing silver rows at s3://cyber-threat-intel-data-lake/silver/cves_clean/: 355726
Updated rows for cves: 355726
Saved staging: s3://cyber-threat-intel-data-lake/silver/_staging/cves/2026-06-07/
Saved Parquet: s3://cyber-threat-intel-data-lake/silver/cves_clean/
Saved CSV: s3

## 8. Final Validation

In [13]:
silver_checks = {
    "assets": paths["assets_silver"],
    "cisa": paths["cisa_silver"],
    "epss": paths["epss_silver"],
    "cves": paths["cves_silver"],
    "cve_affected_products": paths["cve_affected_products_silver"],
}

for name, path in silver_checks.items():
    df = spark.read.parquet(path)
    print(name, "rows:", df.count())
    df.printSchema()

assets rows: 5000
root
 |-- asset_id: string (nullable = true)
 |-- hostname: string (nullable = true)
 |-- vendor: string (nullable = true)
 |-- product: string (nullable = true)
 |-- version: string (nullable = true)
 |-- criticality: string (nullable = true)
 |-- internet_facing: string (nullable = true)

cisa rows: 1612
root
 |-- cve_id: string (nullable = true)
 |-- vendor: string (nullable = true)
 |-- product: string (nullable = true)
 |-- vulnerability_name: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- date_added: date (nullable = true)
 |-- due_date: date (nullable = true)
 |-- known_ransomware_campaign_use: string (nullable = true)
 |-- cwes: string (nullable = true)

epss rows: 676817
root
 |-- cve_id: string (nullable = true)
 |-- epss_score: double (nullable = true)
 |-- percentile: double (nullable = true)
 |-- score_date: date (nullable = true)

cves rows: 355726
root
 |-- cve_id: string (nullable = true)
 |-- description: string (nullab

In [14]:
raw_cves = spark.read.option("multiLine", "true").json(paths["cves_bronze"])

print("=" * 60)
print("BRONZE SCHEMA")
print("=" * 60)

raw_cves.printSchema()

BRONZE SCHEMA
root
 |-- cve: struct (nullable = true)
 |    |-- cisaActionDue: string (nullable = true)
 |    |-- cisaExploitAdd: string (nullable = true)
 |    |-- cisaRequiredAction: string (nullable = true)
 |    |-- cisaVulnerabilityName: string (nullable = true)
 |    |-- configurations: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- nodes: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- cpeMatch: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- criteria: string (nullable = true)
 |    |    |    |    |    |    |    |-- matchCriteriaId: string (nullable = true)
 |    |    |    |    |    |    |    |-- versionEndExcluding: string (nullable = true)
 |    |    |    |    |    |    |    |-- versionEndIncluding: string (nullable = true)
 |    |    |    |    |    |    |    |-- versionSt